In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import random
answers = ['yes', 'probably', 'idk', 'probably not', 'no', 'y', 'probably yes', 'i dont know', 'maybe', 'probably no', 'n']

def load_dataset(dataset_path):
    df = pd.read_csv(dataset_path)
    y = df.iloc[:, 0]        
    X = df.iloc[:, 1:]     
    return X, y
dataset_path = "animal_dataset.csv"
X, y = load_dataset(dataset_path)

In [105]:
def best_feature_to_ask(X, candidate_animals=None, asked_features=None):
    if candidate_animals is None:
        X_sub = X
    else:
        X_sub = X.loc[candidate_animals]
    if asked_features:
        X_sub = X_sub.drop(columns=asked_features)
    best_feature = None
    best_split = 1.0
    for feature in X_sub.columns:
        yes_ratio = (X_sub[feature] == 1).mean()
        split_quality = abs(0.5 - yes_ratio)
        if split_quality < best_split:
            best_split = split_quality
            best_feature = feature
    return best_feature

def update_likelihood(animal_value, new_series, i, answer):
    if answer in ('yes', 'y'):
        new_series.loc[i] += 1 if animal_value == 1 else 0
    elif answer in ('probably', 'probably yes'):
        new_series.loc[i] += 0.75 if animal_value == 1 else 0.25
    if answer in ('i dont know', 'idk', 'maybe'):
        new_series.loc[i] += 0.5
    if answer in ('probably not', 'probably no'):
        new_series.loc[i] += 0.25 if animal_value == 1 else 0.75
    if answer in ('no', 'n'):
        new_series.loc[i] += 0 if animal_value == 1 else 1
    return new_series

def is_confident_enough(new_series, margin=0.5):
    sorted_vals = new_series.sort_values(ascending=False)
    top = sorted_vals.iloc[0]
    second = sorted_vals.iloc[1]
    return (top - second >= margin)

def print_top_likelihoods(new_series, y, top_n=5):
    sorted_vals = new_series.sort_values(ascending=False)
    
    print("\nTop likelihoods:")
    for idx, val in sorted_vals.head(top_n).items():
        print(f"- {y[idx]} : {val:.3f}")
    print()

def check_stuck(top_history, new_series, top_n=3, required_repeats=4):
    current_top = tuple(new_series.sort_values(ascending=False).head(top_n).index)
    if not top_history:
        return False
    if len(top_history) >= required_repeats - 1:
        if all(prev == current_top for prev in top_history[-(required_repeats):]):
            return True
    return False

def most_discriminative_feature(X, candidates, asked_features, useless_features):

    def best_feature_from_list(features):
        best_feature = None
        best_var = -1

        for f in features:
            if f in useless_features:
                continue
            values = X.loc[candidates, f]
            var = values.var()

            if var > best_var and var > 0:
                best_var = var
                best_feature = f

        return best_feature

    asked_list = [f for f in asked_features if f in X.columns and f not in useless_features]
    best = best_feature_from_list(asked_list)

    if best is not None:
        return best

    unasked = [f for f in X.columns if f not in asked_features]
    best = best_feature_from_list(unasked)

    if best is not None:
        return best

    return None

In [115]:
def play_game(X, y):
    likelihood = pd.Series(0.0, index=X.index)
    asked_features = set()
    useless_features = set()
    top_history = []
    
    while True:
        if is_confident_enough(likelihood):
            best_idx = likelihood.idxmax()
            animal = y[best_idx]
            print(f"\nI am confident your animal is: {animal}")
            answer = input(f"Is your animal a {animal}? ").strip().lower()
            if answer in ("yes", "y"):
                print("Yess! I got it right!")
                return
            else:
                likelihood = likelihood.drop(best_idx)
                continue
        if len(likelihood) > 1 and check_stuck(top_history, likelihood, top_n=3, required_repeats=4) and len(asked_features) > 5:
            max_value = likelihood.max()
            top_candidates = likelihood[likelihood == max_value].index
            fallback_feature = most_discriminative_feature(X, top_candidates, asked_features, useless_features)
            if fallback_feature:
                print("\nIt seems I'm not getting new information from your answers. Please let me ask you this specific question again:")
                feature = fallback_feature
                useless_features.add(feature)
            else:
                animal_names = tuple(y[top_candidates])
                print(f"I cannot differentiate further. My best guess is: {animal_names}")
                return
        else:
            top_candidates = likelihood[likelihood == likelihood.max()].index
            feature = best_feature_to_ask(X, candidate_animals=top_candidates, asked_features=asked_features)
        asked_features.add(feature)
        answer = input(f"{feature}? ").strip().lower()
        while answer not in answers:
            answer = input("I didn't understand. Say it again: ").strip().lower()
        for i in likelihood.index:
            likelihood = update_likelihood(X.loc[i, feature], likelihood, i, answer)

        #print_top_likelihoods(likelihood, y)
        current_top = tuple(likelihood.sort_values(ascending=False).head(3).index)
        top_vals = likelihood.sort_values(ascending=False)
        max_val = top_vals.iloc[0]
        tied = list(top_vals[top_vals == max_val].index)
        if len(tied) > 3:
            current_top = tuple(random.sample(tied, 3))
        if len(asked_features) > 4:
            top_history.append(current_top)
        if len(top_history) > 5: 
            top_history.pop(0)
play_game(X, y)

Is your animal native to Australia?  no
Is your animal a herbivore?  no
Has your animal fur?  yes
Is your animal a carnivore?  yes
Is your animal larger than a human?  yes
Is your animal native to Africa?  idk
Has your animal stripes?  idk



It seems I'm not getting new information from your answers. Please let me ask you this specific question again:


Has your animal stripes?  idk



It seems I'm not getting new information from your answers. Please let me ask you this specific question again:


Is your animal native to Africa?  idk


I cannot differentiate further. My best guess is: ('Lion', 'Tiger')
